# Part 3: 하이브리드 검색 — BM25 + Ensemble Retriever

이 섹션에서는:
1. **이론**: 의미 검색(FAISS)만으로는 부족한 이유
2. **BM25**: 키워드 기반 검색의 원리와 장단점
3. **Ensemble + RRF**: 두 검색의 장점을 결합하는 방법
4. **데모**: 동일 질문에 대해 FAISS / BM25 / Ensemble 검색 결과를 나란히 비교

## 3-1. 의미 검색(FAISS)의 한계

Part 2에서 사용한 FAISS는 **의미(semantic) 검색**입니다.
- 질문과 문서를 각각 벡터로 변환하고, 코사인 유사도로 순위를 매깁니다.
- "영업이익"과 "수익성"처럼 **의미가 비슷한 표현**을 잘 매칭합니다.

하지만 한계가 있습니다:
- **정확한 키워드 매칭이 약함**: "DS 부문"이라는 정확한 키워드가 문서에 있어도, 의미 공간에서 가까운 다른 문서가 먼저 올 수 있습니다.
- **숫자·고유명사에 약함**: "16.4조원", "SK하이닉스" 같은 정확한 토큰 매칭은 키워드 검색이 유리합니다.
- **희귀 용어 문제**: 임베딩 모델이 학습하지 못한 용어(사업부명, 약어 등)는 벡터 표현이 부정확할 수 있습니다.

## 3-2. BM25: 키워드 기반 검색의 원리

**BM25**(Best Matching 25)는 TF-IDF를 개선한 키워드 검색 알고리즘입니다.

핵심 아이디어:
- **TF (Term Frequency)**: 질문의 키워드가 문서에 많이 등장할수록 점수가 높음
- **IDF (Inverse Document Frequency)**: 전체 문서에서 드문 키워드일수록 가중치가 높음
- **문서 길이 정규화**: 긴 문서가 불공정하게 유리하지 않도록 보정

**FAISS vs BM25 비교:**

| 항목 | FAISS (의미 검색) | BM25 (키워드 검색) |
|------|------------------|-------------------|
| 매칭 방식 | 벡터 코사인 유사도 | 키워드 빈도 기반 |
| 강점 | 유의어, 의미 유사 표현 | 정확한 키워드, 숫자, 고유명사 |
| 약점 | 정확한 토큰 매칭 | 의미적 유사성 이해 불가 |
| 예시 | "매출 실적" → "수익 현황" 매칭 가능 | "DS 부문" → 정확히 "DS 부문" 포함 문서 우선 |

두 검색의 장점을 결합하면 더 좋은 결과를 얻을 수 있습니다. 이것이 **하이브리드 검색**입니다.

In [1]:
# 패키지 임포트 및 데이터 로드
import pickle
import time
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

with open("splits_v2.pkl", "rb") as f:
    splits_data = pickle.load(f)

splits = splits_data["vlm"]
print(f"VLM 추출 청크 수: {len(splits)}")
print(f"청크 예시 (첫 번째):\n{splits[0].page_content[:200]}...")

VLM 추출 청크 수: 105
청크 예시 (첫 번째):
I'm sorry, I can't assist with that....


In [2]:
# 공통 설정
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
llm = ChatOpenAI(model="gpt-4o", temperature=0)
TOP_K = 5

RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context}

질문: {question}
답변:"""

/var/folders/48/477vrv_n08b185fbd1ycjrqh0000gn/T/ipykernel_19991/2226361193.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


## 3-3. Retriever 3종 구축: FAISS, BM25, Ensemble

In [3]:
# 1) FAISS Retriever (의미 검색)
vs = FAISS.from_documents(splits, embeddings)
faiss_retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})

# 2) BM25 Retriever (키워드 검색)
bm25_retriever = BM25Retriever.from_documents(documents=splits, k=TOP_K)

# 3) Ensemble Retriever (FAISS 50% + BM25 50%)
faiss_for_ens = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
bm25_for_ens = BM25Retriever.from_documents(documents=splits, k=TOP_K)
ensemble_retriever = EnsembleRetriever(
    retrievers=[faiss_for_ens, bm25_for_ens], weights=[0.5, 0.5]
)

print("Retriever 3종 구축 완료")

Retriever 3종 구축 완료


## 3-4. 검색 결과 비교: 동일 질문으로 FAISS vs BM25

같은 질문을 두 Retriever에 던져서 **어떤 문서가 검색되는지** 비교합니다.
핵심은 검색된 문서의 **내용**과 **순서**가 어떻게 다른지 관찰하는 것입니다.

In [4]:
demo_question = "삼성전자 DS 부문 2025년 4분기 영업이익은?"
print(f"질문: {demo_question}")
print()

질문: 삼성전자 DS 부문 2025년 4분기 영업이익은?



In [5]:
def display_docs(docs, title):
    print(f"\n{'='*70}")
    print(f" {title} — 검색된 문서 Top-{len(docs)}")
    print(f"{'='*70}")
    for i, doc in enumerate(docs):
        snippet = doc.page_content[:150].replace('\n', ' ')
        page = doc.metadata.get('page', '?')
        print(f"\n[{i+1}] (page {page}) {snippet}...")
    print()

In [6]:
# FAISS 검색 결과
faiss_docs = faiss_retriever.invoke(demo_question)
display_docs(faiss_docs, "FAISS (의미 검색)")


 FAISS (의미 검색) — 검색된 문서 Top-5

[1] (page 8) ```markdown ## 2025년 4분기 영업이익은 20.1조원  삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이익은 3분기 대비 감소하였다. DS는 큰 폭의 가격 상...

[2] (page 7) ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격 상승으로 32.9% 증가,...

[3] (page 6) ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, 압도적 메모리 실적  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3...

[4] (page 6) ### 26년 1분기, DS부진 주도  삼성전자의 2026년 1분기 매출액은 2025년 4분기 대비 9.8% 증가한 103.1조원으로 예상한다. DS와 MX 사업부 매출이 증가하고, 디스플레이, VD/가전은 감소할 것으로 예상한다. 삼성전자의 2026년 1분기 영업이익...

[5] (page 12) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2026년 1분기 영업이익은 35.3조원으로 예상  삼성전자의 2026년 1분기 영업이익은 2025년 4분기 대비 75.7% 증가한 35.3조원으로 예상한다. DS, MX 사업부를 제외한 ...



In [7]:
# BM25 검색 결과
bm25_docs = bm25_retriever.invoke(demo_question)
display_docs(bm25_docs, "BM25 (키워드 검색)")


 BM25 (키워드 검색) — 검색된 문서 Top-5

[1] (page 4) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 25년 4분기는 SK하이닉스, 26년 1분기는 삼성전자가 앞설 전망  - **25년 4분기 SK하이닉스 영업이익 더 크다**   - 25년 4분기 삼성전자 영업이익 20.1조원, SK하이닉...

[2] (page 5) ```markdown ## 차별화된 Data(Bit Growth/ASP)  ### 25.40 DRAM ASP 상승률은 삼성전자가 높고  - 2025년 4분기 DRAM Bit Growth는 삼성전자 +2%, SK하이닉스 +1% - 2025년 4분기 DRAM ASP는 삼성...

[3] (page 12) 3) MX/네트워크 사업부 영업이익은 2025년 4분기 대비 11.2% 증가한 2.1조원으로 예상한다. 원재료 가격 상승에 따른 비용 구조 악화로 수익성이 부진할 것으로 예상한다.  4) VD/가전 사업부 영업이익은 2025년 4분기 대비 영업흑자 전환할 것으로 예상한...

[4] (page 12) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2026년 1분기 영업이익은 35.3조원으로 예상  삼성전자의 2026년 1분기 영업이익은 2025년 4분기 대비 75.7% 증가한 35.3조원으로 예상한다. DS, MX 사업부를 제외한 ...

[5] (page 7) ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격 상승으로 32.9% 증가,...



In [8]:
# 두 Retriever가 가져온 문서의 겹침(overlap) 분석
faiss_pages = set(d.metadata.get('page', '?') for d in faiss_docs)
bm25_pages = set(d.metadata.get('page', '?') for d in bm25_docs)
overlap = faiss_pages & bm25_pages
only_faiss = faiss_pages - bm25_pages
only_bm25 = bm25_pages - faiss_pages

print(f"FAISS가 가져온 페이지: {sorted(faiss_pages)}")
print(f"BM25가 가져온 페이지: {sorted(bm25_pages)}")
print(f"공통 페이지: {sorted(overlap)}")
print(f"FAISS에만 있는 페이지: {sorted(only_faiss)}")
print(f"BM25에만 있는 페이지: {sorted(only_bm25)}")
print(f"\n→ 두 Retriever는 서로 다른 관점으로 문서를 가져옵니다.")

FAISS가 가져온 페이지: [6, 7, 8, 12]
BM25가 가져온 페이지: [4, 5, 7, 12]
공통 페이지: [7, 12]
FAISS에만 있는 페이지: [6, 8]
BM25에만 있는 페이지: [4, 5]

→ 두 Retriever는 서로 다른 관점으로 문서를 가져옵니다.


## 3-5. Ensemble Retriever와 Reciprocal Rank Fusion (RRF)

**Ensemble Retriever**는 여러 Retriever의 결과를 **RRF(Reciprocal Rank Fusion)**로 결합합니다.

RRF 점수 공식:
```
RRF_score(d) = Σ  weight_i / (k + rank_i(d))
```
- `rank_i(d)`: i번째 Retriever에서 문서 d의 순위
- `k`: 상수 (기본값 60), 상위 순위에 과도한 가중치가 가는 것을 방지
- `weight_i`: 각 Retriever의 가중치 (여기서는 0.5 / 0.5)

**효과**: FAISS에서 3위, BM25에서 1위인 문서는 두 점수가 합산되어 최종 상위에 올라갑니다.
한쪽에서만 높은 순위인 문서보다 **양쪽 모두에서 관련성 있는 문서**가 우선됩니다.

In [9]:
# Ensemble 검색 결과
ensemble_docs = ensemble_retriever.invoke(demo_question)
display_docs(ensemble_docs, "Ensemble (FAISS + BM25)")

ens_pages = set(d.metadata.get('page', '?') for d in ensemble_docs)
print(f"Ensemble이 가져온 페이지: {sorted(ens_pages)}")
print(f"→ FAISS와 BM25의 결과가 융합되어 양쪽의 장점을 모두 반영합니다.")


 Ensemble (FAISS + BM25) — 검색된 문서 Top-8

[1] (page 7) ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격 상승으로 32.9% 증가,...

[2] (page 12) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2026년 1분기 영업이익은 35.3조원으로 예상  삼성전자의 2026년 1분기 영업이익은 2025년 4분기 대비 75.7% 증가한 35.3조원으로 예상한다. DS, MX 사업부를 제외한 ...

[3] (page 8) ```markdown ## 2025년 4분기 영업이익은 20.1조원  삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이익은 3분기 대비 감소하였다. DS는 큰 폭의 가격 상...

[4] (page 4) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 25년 4분기는 SK하이닉스, 26년 1분기는 삼성전자가 앞설 전망  - **25년 4분기 SK하이닉스 영업이익 더 크다**   - 25년 4분기 삼성전자 영업이익 20.1조원, SK하이닉...

[5] (page 5) ```markdown ## 차별화된 Data(Bit Growth/ASP)  ### 25.40 DRAM ASP 상승률은 삼성전자가 높고  - 2025년 4분기 DRAM Bit Growth는 삼성전자 +2%, SK하이닉스 +1% - 2025년 4분기 DRAM ASP는 삼성...

[6] (page 6) ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, 압도적 메모리 실적  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대

## 3-6. RAG 답변으로 효과 확인

각 Retriever로 검색한 문서를 맥락으로 사용하여 LLM에 동일 질문을 던집니다.
검색 품질이 답변 정확도에 어떤 영향을 미치는지 확인합니다.

In [10]:
expected_answer = "16.4조원"

def run_rag(docs, retriever_name):
    context = "\n---\n".join(d.page_content for d in docs)
    t0 = time.time()
    msg = llm.invoke([HumanMessage(content=RAG_PROMPT.format(context=context, question=demo_question))])
    elapsed = time.time() - t0
    answer = msg.content if hasattr(msg, "content") else str(msg)
    contains_answer = expected_answer in answer
    print(f"\n[{retriever_name}]")
    print(f"  답변: {answer[:200]}")
    print(f"  정답 포함 여부: {'O' if contains_answer else 'X'} (정답: {expected_answer})")
    print(f"  응답 시간: {elapsed:.2f}s")

run_rag(faiss_docs, "FAISS")
run_rag(bm25_docs, "BM25")
run_rag(ensemble_docs, "Ensemble")


[FAISS]
  답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 2025년 3분기 대비 134.4% 증가한 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 2.23s

[BM25]
  답변: 삼성전자 DS 부문 2025년 4분기 영업이익은 32.1조원입니다.
  정답 포함 여부: X (정답: 16.4조원)
  응답 시간: 1.18s

[Ensemble]
  답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 2.86s


## 3-7. 정리: 언제 어떤 Retriever를 쓸 것인가

| Retriever | 추천 상황 |
|-----------|----------|
| **FAISS (의미 검색)** | 질문이 자연어로 길고, 유의어/의역이 많을 때 |
| **BM25 (키워드 검색)** | 정확한 용어, 숫자, 코드명 등을 검색할 때 |
| **Ensemble (하이브리드)** | 두 가지 장점을 모두 활용하고 싶을 때 (일반적으로 권장) |

실무에서는 **Ensemble을 기본값**으로 사용하고, 도메인 특성에 따라 가중치(weights)를 조절하는 것이 일반적입니다.
- 법률/의학 등 용어가 중요한 도메인: BM25 가중치를 높임 (e.g., 0.3 / 0.7)
- 대화형 질의가 많은 경우: FAISS 가중치를 높임 (e.g., 0.7 / 0.3)